# SOLUCIÓN PROBLEMA 1 - INTER 2 Probabilidad y Estadística - iMAT-ICAI - 7 de marzo 2022

# Vale el 40% de la nota del examen
---

NO CAMBIES EL NOMBRE AL NOTEBOOK, SIMPLEMENTE ESCRIBE AQUÍ TUS DATOS
### Nombre: RELLENAR
### Apellidos: RELLENAR
---

### Recientemente se ha encontrado junto a una antigua tumba en Egipto un almacén con toneladas de semillas de trigo, de la época de Ramsés II. Se quiere estudiar las características de este antiguo trigo, para lo que se coge con cuidado una muestra representativa con 5000 semillas. En concreto se quiere estudiar el rendimiento del trigo encontrado, pues podría ser mejor que el que tenemos en la actualidad.

### El rendimiento de una semilla concreta de trigo se mide a partir de las semillas que se obtienen en la primera cosecha de la planta que ha crecido a partir de dicha semilla, cultivada en condiciones óptimas. Cuantas más semillas se obtengan a partir de una semilla, más rendimiento habrá tenido. En concreto, el rendimiento X se calcula dividiendo el peso de las semillas obtenidas entre el peso de la semilla plantada. El mejor trigo actual tiene un rendimiento medio de 1123.

### El equipo internacional de científicos encargado de realizar el estudio, del cual formas parte, ha cultivado en condiciones óptimas las 5000 semillas de la muestra, calculando con cuidado para cada semilla su rendimiento.

Responde las preguntas utilizando instrucciones print() para mostrar claramente tu respuesta en la ejecución de la celda, explicando cómo lo resuelves. Utiliza comentarios en tu código para facilitar su lectura.

### Importamos todos los paquetes aquí por comodidad

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
from scipy import optimize as opt  # para estimación por máxima verosimilitud

### a) Los rendimientos de cada semilla de la muestra estaban almacenados en el portátil de un joven científico. Lamentablemente, ayer se estaba tomando un café y derramó toda la taza sobre el mismo. Sí, no tenía copia de seguridad de ningún tipo, a pesar de las recomendaciones de la jefa de proyecto. Todavia no le ha comentado lo del "accidente" del portátil, a pesar de que tiene un correo suyo solicitándole urgentemente un intervalo de confianza para el rendimiento medio de la muestra con un nivel de confianza del 5%.

### Por suerte, el científico se acuerda perfectamente de la media del rendimiento de la muestra y de su desviación típica: x_med = 1123.58 y x_std = 13.21, los primeros números de la serie de Fibonacci.

### Ayuda a tu compañero de equipo y calcula, a partir de los valores x_med y x_std anteriores, el intervalo de confianza que le está reclamando la jefa. Justifica la respuesta. (2 puntos)

In [ ]:
# Respuesta

# Datos de partida (muestra)
x_med = 1123.58
x_std = 13.21
n = 5000

se = x_std / np.sqrt(n)

print(f'Se tiene una muestra de tamaño {n} de la variable X, se ha calculado la media y la std de X en la muestra.')
print('Como n es grande, se puede asumir que la media muestral sigue una normal de parámetros:')
print(f' - Media: {x_med}')
print(f' - Desviación típica x_std/raiz(n) = {x_std} / {np.sqrt(n)} = {se}') 
   
# creamos la v.a. MEDIA MUESTRAL como una t de student
#    De la ayuda de scipy.stats.t :
#    To shift and/or scale the distribution use the loc and scale parameters. 
#    Specifically, t.pdf(x, df, loc, scale) is identically equivalent to t.pdf(y, df) / scale with y = (x - loc) / scale.
Xmedia = stats.t(n-1, loc = x_med, scale = se)  

#Para calcular un IC de la media de X usamos la distribución teórica de la media muestral
gama = 0.95
alfa = 1 - gama
lim_inf_95 = Xmedia.ppf(alfa/2)
lim_sup_95 = Xmedia.ppf(1-alfa/2)

print(f'El intervalo al {100*gama}% para la media del rendimiento del trigo antiguo es: [{lim_inf_95}, {lim_sup_95}]')

# Posible mejora del intervalo de confianza
print('\nEs posible refinar un poco el resultado usando la varianza muestral corregida para evitar el sesgo '
      '(será pequeño porque el tamaño muestral es grande, 5000):')
x_std_corr = np.sqrt((n/(n-1)) * x_std**2) 
print(f'x_std = {x_std} -> x_std_corr = {x_std_corr}')

Xmedia2 = stats.t(n-1, loc = x_med, scale = x_std_corr / np.sqrt(n))  
print(f'El intervalo al {100*gama}% para la media del rendimiento del trigo antiguo con la correción es: '
      f'[{Xmedia2.ppf(alfa/2)}, {Xmedia2.ppf(1-alfa/2)}]')


# IC calculado utilizando una normal
print('\nEl IC también se puede estimar suponiendo una normal en lugar de una t-Student, ya que n es grande '
      '(el resultado es muy parecido):')

Xmedia3 = stats.norm(loc = x_med, scale = se)  
print(f'El intervalo al {100*gama}% para la media del rendimiento del trigo antiguo según una normal es: '
      f'[{Xmedia3.ppf(alfa/2)}, {Xmedia3.ppf(1-alfa/2)}]')


### b) Representa en la misma gráfica, la función de densidad de la distribución del estadistico utilizado para calcular el intervalo de confianza del apartado anterior, así como el intervalo de confianza construido. Indica claramente el área asociada al nivel de confianza utilizado. (0.5 puntos)

In [ ]:
# Respuesta

# puntos para representar la distribución y el intervalo
xs = np.linspace(Xmedia.ppf(0.00001), Xmedia.ppf(1-0.00001), 1000)
xIC95 = np.linspace(lim_inf_95, lim_sup_95, 1000)

plt.figure(1, figsize=(15, 6))
plt.plot(xs, Xmedia.pdf(xs), linewidth = 4, label=f'Distribución Media muestral (según una t)')
plt.fill_between( xIC95, Xmedia.pdf(xIC95), color="green", alpha=0.3, label=f'Probabilidad asociada ($\gamma=${gama})')
plt.vlines(x_med, 0, np.max(Xmedia.pdf(xs)), color='black', label=f'Estimación puntual media {x_med}')
plt.errorbar(x_med, -0.01, xerr = np.array([[x_med - lim_inf_95], [lim_sup_95 - x_med]]), fmt = 'o', label = f'IC [{lim_inf_95}, {lim_sup_95}] al {gama*100}%')
plt.title('Distribución del estimador media muestral de X e IC')
plt.tight_layout()
plt.legend()
plt.show()

### c) En cuanto la jefa de proyecto se entera del intervalo calculado al 95%, le quita importancia al "accidente" con el portátil y rápidamente solicita que se calcule un nuevo intervalo de confianza para la media del rendimiento del trigo antiguo, pero al 99%. Repite el cálculo con ese nuevo intervalo de confianza. ¿Tiene un ancho mayor? Justifica la respuesta.  (1 punto)

In [ ]:
# Respuesta

#Para calcular un IC de la media de X usamos la distribución teórica de la media muestral
gama = 0.99
alfa = 1 - gama
lim_inf_99 = Xmedia.ppf(alfa/2)
lim_sup_99 = Xmedia.ppf(1-alfa/2)

print(f'El intervalo al {100*gama}% para la media del rendimiento del trigo antiguo es: [{lim_inf_99}, {lim_sup_99}]')

print('Claro que tiene un ancho mayor, es la única forma de poder aumentar la confianza en el intervalo calculado')

### d) Representa en la misma gráfica, la función de densidad de la distribución del estadistico utilizado para calcular los intervalos de confianza de los dos apartados anteriores, así como dichos intervalos de confianza. (0.5 puntos)

In [ ]:
# Respuesta

# puntos para representar la distribución y el intervalo
xs = np.linspace(Xmedia.ppf(0.00001), Xmedia.ppf(1-0.00001), 1000)
xIC99 = np.linspace(lim_inf_99, lim_sup_99, 1000)

plt.figure(1, figsize=(15, 6))
plt.plot(xs, Xmedia.pdf(xs), linewidth = 4, label=f'Distribución Media muestral (según una t)')
plt.fill_between( xIC95, Xmedia.pdf(xIC95), color="green", alpha=0.3, label=f'Probabilidad asociada ($\gamma=${0.95})')
plt.fill_between( xIC99, Xmedia.pdf(xIC99), color="orange", alpha=0.4, label=f'Probabilidad asociada ($\gamma=${0.99})')
plt.vlines(x_med, 0, np.max(Xmedia.pdf(xs)), color='black', label=f'Estimación puntual media {x_med}')
plt.errorbar(x_med, -0.04, xerr = np.array([[x_med - lim_inf_95], [lim_sup_95 - x_med]]), fmt = 'o', label = f'IC [{lim_inf_95}, {lim_sup_95}] al {95}%')
plt.errorbar(x_med, -0.08, xerr = np.array([[x_med - lim_inf_99], [lim_sup_99 - x_med]]), fmt = 'o', label = f'IC [{lim_inf_99}, {lim_sup_99}] al {99}%')
plt.title('Distribución del estimador media muestral de X e IC')
plt.tight_layout()
plt.xlabel('Rendimiento del trigo')
plt.legend()
plt.show()

### e) Conocido el nuevo intervalo, la jefa no puede ocultar su alegría. Grita "¡¡ LO TENEMOS !!". Explica por qué está tan contenta. (1 punto)

In [ ]:
# Respuesta

print('El intervalo de confianza al 99% no contiene a la media del trigo actual, que vale 1123, por lo que todo apunta a que el trigo encontrado '
      'tiene un rendimiento significativamente mayor.')

### La jefa, con dilatada experiencia, tenía contratado un seguro de accidentes que cubre la recuperación de los datos del disco del portátil. Una semana después recibe un e-mail en el que se adjunta el fichero muestraRecuperada5000semillas.csv

In [ ]:
# cargamos los datos de la muestra disponible

df_orig = pd.read_csv('muestraRecuperada5000semillas.csv')
print('Tamaño de df_orig con los datos cargados:', df_orig.shape)

# datos de la muestra para trabajar
x = df_orig.x.tolist()    #   <-------------------------------- x (valores en la muestra)
n = len(x)                #   <-------------------------------- n (tamaño de la muestra)
 
# los datos de las semillas
print('Valores obtenidos para cada semilla:')
plt.figure(1, figsize=(15, 4))
plt.plot(x)
plt.xlabel('Semilla')
plt.ylabel('Rendimiento')
plt.grid()
plt.show()

# observamos su distribución
print('Distribución del rendimiento en la muestra de semillas:')
plt.figure(1, figsize=(15, 4))
plt.hist(x, bins = 40);plt.xlabel('x');plt.title(f'Muestra (n={n})')
plt.show()

### f) Sabiendo que se quiere demostrar que el trigo antiguo tiene más rendimiento que el actual, plantea el contraste lógico a realizar, indicando las hipótesis H0 y H1. Resuelve el contraste de forma detallada a partir de la muestra disponible, con un nivel de significación del 1%. Calcula el valor del estadístico de contraste en la muestra, los valores críticos y el p-valor asociado. Razona utilizando por separado los valores críticos y el p-valor para tener más seguridad en lo que haces. No olvides concluir claramente. (4 puntos)

In [ ]:
# Respuesta

mu_0 = 1123

# datos de la muestra
n = len(x)
x_media = np.mean(x)
sx = np.std(x, ddof=1) # dividiendo por n-1
print(f'La media de la muestra es {x_media} y su desv. típica {sx}. Parece que el científico no había '
      'calculado perfectamente dichos valores, o los recordaba mal.')


# Contraste de hipótesis para la media de X
print('\nEl contraste lógico a realizar es unilateral ("mayor que"):')
print(f' H0: mu_X = {mu_0}')
print(f' H1: mu_X > {mu_0}')

t0 = (x_media - mu_0) / (sx/np.sqrt(n))

# crea la distribución del estadístico de contraste
df = n -1 # grados de libertad
T0 = stats.t(df)

# calcula el pvalor (acorde a H1: mu > mu_0)
pvalor = 1 - T0.cdf(t0)

# determina los valores críticos que definen la región de rechazo (solo hay una situada a la derecha)
alfa = 0.01  # nivel de significación
t_crit_de = T0.ppf(1-alfa)


print(f'El valor observado del estadistico de contraste T0 es {t0}, un valor grande considerando '
      f'que el error estandar de T0 es {(sx/np.sqrt(n))}')

print(f'La región de rechazo está situada a la derecha, por lo que solo hay un valor crítico, que al 1% vale {t_crit_de}')
print(f'Como t0 está dentro de la región de rechazo ({t0} > {t_crit_de}), se puede rechazar la hipótesis nula al 1%.')

print(f'El p-valor asociado es igual a {pvalor}.')
print(f'Como el pvalor es menor que alfa ({pvalor} <{alfa}), hay que recharzar HO al 1% de significación, en favor de H1.')

print('Por tanto, el test de hipótesis ha resultado significativo. Se puede concluir al 1% de nivel de significación que el trigo antiguo '
      'encontrado tiene un rendimiento mayor que el actual.')

In [ ]:
# Gráficamente para terminar de confirmar que todo está bien

# puntos para representar la distribución y las regiones de aceptación y rechazo
xmin = T0.ppf(0.000001)
xmax = T0.ppf(1-0.000001)
xs = np.linspace(xmin, xmax, 1000)
xRA = np.linspace(xmin, t_crit_de, 1000)
xRRde = np.linspace(t_crit_de, xmax, 1000)

plt.figure(1, figsize=(20, 6))
plt.plot(xs, T0.pdf(xs), linewidth = 4, label=f'Distribución T0 si H0 es cierta')

plt.fill_between( xRA, T0.pdf(xRA), color="green", alpha=0.6, label=f'Probabilidad asociada RA = {1-alfa}')
plt.fill_between( xRRde, T0.pdf(xRRde), color="red", alpha=0.6, label=f'P(T0>{t_crit_de}) = {alfa}')

plt.vlines(t0, 0, np.max(T0.pdf(xs)), color='black', label=f't0 = {t0}')

plt.hlines(-0.02, xmin, t_crit_de, color='green', label = f'RA: [-inf,{t_crit_de}] al {alfa*100}%')
plt.hlines(-0.02, t_crit_de, xmax, color='red', linewidth = 6, label = f'RR')

plt.title('Distribución del estimador de contraste')
plt.xlabel('T0')
plt.legend()
plt.show()

print('Es evidente que hay que rechazar HO')

### g) Resuelve el contraste anterior de forma directa y comprueba tu resultado.    (1 punto)

In [ ]:
# Respuesta

# tioo de laternativa
tipo_alternativa =  'greater' # alternativas posibles: ‘two-sided’, ‘less’, ‘greater’

# calculamos el estadístico en la muestra t0 y su pvalor
res = stats.ttest_1samp(x, popmean = mu_0, alternative = tipo_alternativa) 
print('Resultado: ', res)

t0 = res.statistic
pvalor = res.pvalue

print(f'El valor observado del estadistico de contraste es {t0}, con un p-valor asociado igual a {pvalor}.')
print('Los resultados son prácticamente iguales a los calculados paso a paso en el apartado anterior.')
